# Notebook 02 — Silver Layer Cleaning & Joining

## What this notebook does
Takes the 9 raw Bronze tables and produces ONE clean analytical table.
One row = one delivered order with all customer, payment, review,
and item data attached.

## Why Silver exists
Bronze data is raw and messy:
- Dates stored as strings, not timestamps
- customer_id is not the real person identifier
- Payments and items are separate tables that need joining
- Cancelled/pending orders distort purchase behavior analysis
Silver fixes all of this in one place so Gold never has to.

## Source → Target
Source : brazilian.bronze.* (9 managed Delta tables)
Target : brazilian.silver.customer_orders (1 managed Delta table)

## Key decisions made in this notebook
1. customer_unique_id used throughout — never customer_id
2. Filter to delivered orders only — cancelled ≠ purchase relationship
3. Dates cast explicitly from string to TimestampType
4. Payments aggregated to order level before joining
5. Null review scores kept — not dropped — LEFT JOIN

In [0]:
# ── SETUP ─────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark.sql("USE CATALOG brazilian")

# Define source and target clearly at the top
# Any future engineer reading this knows immediately
# where data comes from and where it goes
SOURCE_CATALOG = "brazilian"
SOURCE_SCHEMA  = "bronze"
TARGET_TABLE   = "brazilian.silver.customer_orders"

print(f"Catalog  : {SOURCE_CATALOG}")
print(f"Reading  : {SOURCE_CATALOG}.{SOURCE_SCHEMA}.*")
print(f"Writing  : {TARGET_TABLE}")
print()
print("S3 : not used in this notebook or any notebook after this")

In [0]:
# ── DATA QUALITY INVESTIGATION ────────────────────────────────
#
# WHY THIS CELL EXISTS:
# Never transform data you haven't investigated first.
# These findings become your interview talking points.
# When an interviewer asks "what data quality issues did you find?"
# you read directly from this cell's output.
#
# Run this cell, read the output, understand what you found,
# THEN proceed to the transformation in Cell 4.

print("=" * 65)
print("DATA QUALITY INVESTIGATION — Brazilian Olist Dataset")
print("=" * 65)

# ── FINDING 1: The most important data trap in this dataset ───
# customer_id is generated per order in Olist's system
# The same person placing 3 orders gets 3 different customer_ids
# If you use customer_id for RFM, frequency is always 1 for every customer
# customer_unique_id is the stable identifier for a real person
r1 = spark.sql("""
    SELECT
        COUNT(DISTINCT customer_id)        AS customer_ids_total,
        COUNT(DISTINCT customer_unique_id) AS real_unique_people
    FROM brazilian.bronze.olist_customers
""").collect()[0]

print(f"\n[FINDING 1] The customer_id vs customer_unique_id trap")
print(f"  customer_id (generated per order) : {r1.customer_ids_total:,}")
print(f"  customer_unique_id (real person)  : {r1.real_unique_people:,}")
print(f"  Difference                        : {r1.customer_ids_total - r1.real_unique_people:,}")
print(f"  Using wrong column inflates count by : {round(r1.customer_ids_total/r1.real_unique_people,2)}x")
print(f"  Fix: ALL downstream code uses customer_unique_id only")

# ── FINDING 2: Order status breakdown ────────────────────────
# We need to know how many orders are actually delivered
# vs cancelled vs processing before we filter
print(f"\n[FINDING 2] Order status breakdown")
spark.sql("""
    SELECT
        order_status,
        COUNT(*) AS order_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM brazilian.bronze.olist_orders
    GROUP BY order_status
    ORDER BY order_count DESC
""").show()
print("  Fix: filter to delivered only")
print("  Business reason: cancelled/processing orders do not represent")
print("  a completed purchase relationship with the customer")
print("  Including them would make churned customers look active")

# ── FINDING 3: Null delivery timestamps ──────────────────────
# order_delivered_customer_date is null for non-delivered orders
# This is expected — not a data error
# Our delivered filter eliminates these nulls automatically
r3 = spark.sql("""
    SELECT
        COUNT(*) AS total_orders,
        SUM(CASE WHEN order_delivered_customer_date IS NULL
            THEN 1 ELSE 0 END) AS null_delivery_dates
    FROM brazilian.bronze.olist_orders
""").collect()[0]

print(f"\n[FINDING 3] Null delivery timestamps")
print(f"  Total orders            : {r3.total_orders:,}")
print(f"  Null delivery dates     : {r3.null_delivery_dates:,}")
print(f"  Fix: expected for non-delivered — eliminated by delivered filter")
print(f"  No special handling needed")

# ── FINDING 4: Date columns stored as strings ────────────────
# inferSchema reads timestamps as strings in this dataset
# We must cast them explicitly to TimestampType in Silver
print(f"\n[FINDING 4] Date columns are stored as strings in Bronze")
spark.sql("""
    SELECT
        order_purchase_timestamp,
        TYPEOF(order_purchase_timestamp) AS data_type
    FROM brazilian.bronze.olist_orders
    LIMIT 2
""").show(truncate=False)
print("  Fix: CAST each date column AS TIMESTAMP explicitly in Silver")

# ── FINDING 5: Payment and review coverage ───────────────────
r5 = spark.sql("""
    SELECT
        COUNT(DISTINCT o.order_id)  AS delivered_orders,
        COUNT(DISTINCT p.order_id)  AS orders_with_payments,
        COUNT(DISTINCT r.order_id)  AS orders_with_reviews
    FROM brazilian.bronze.olist_orders o
    LEFT JOIN brazilian.bronze.olist_payments p ON o.order_id = p.order_id
    LEFT JOIN brazilian.bronze.olist_reviews  r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
""").collect()[0]

print(f"\n[FINDING 5] Payment and review coverage on delivered orders")
print(f"  Delivered orders        : {r5.delivered_orders:,}")
print(f"  Orders with payments    : {r5.orders_with_payments:,}  ({round(r5.orders_with_payments/r5.delivered_orders*100,1)}%)")
print(f"  Orders with reviews     : {r5.orders_with_reviews:,}  ({round(r5.orders_with_reviews/r5.delivered_orders*100,1)}%)")
print(f"  Fix: LEFT JOIN on both — keep all delivered orders")
print(f"  Null payment_value → COALESCE to 0")
print(f"  Null review_score  → kept as null intentionally")

In [0]:
# ── BUILD SILVER TABLE ────────────────────────────────────────
#
# WHY SPARK SQL HERE instead of PySpark DataFrame API:
# Multi-table JOINs are cleaner and easier to read in SQL
# An interviewer can look at this query and immediately understand
# the business logic — that is the point of Silver documentation
#
# WHAT THIS QUERY DOES:
# Takes 5 of the 9 bronze tables and joins them into one clean row
# per delivered order. Products, sellers, and geolocation are
# available in the extended analysis notebook.
#
# JOIN STRATEGY:
# - INNER JOIN on customers: every order MUST have a customer record
# - LEFT JOIN on payments:   some orders may have no payment row
# - LEFT JOIN on reviews:    ~75% of orders have reviews, keep all
# - LEFT JOIN on items:      aggregate product items to order level

customer_orders_silver = spark.sql("""
    SELECT

        -- ── CUSTOMER IDENTITY ─────────────────────────────
        -- NEVER use customer_id here — see Finding 1
        -- customer_unique_id is the stable real-person identifier
        c.customer_unique_id,
        c.customer_city,
        c.customer_state,

        -- ── ORDER CORE ────────────────────────────────────
        o.order_id,
        o.order_status,

        -- ── DATES: explicitly cast string to timestamp ────
        -- inferSchema left these as StringType in Bronze
        -- We cast them here once — Silver and Gold never deal with strings
        CAST(o.order_purchase_timestamp       AS TIMESTAMP) AS purchase_ts,
        CAST(o.order_delivered_customer_date  AS TIMESTAMP) AS delivered_ts,
        CAST(o.order_estimated_delivery_date  AS TIMESTAMP) AS estimated_delivery_ts,

        -- ── PAYMENT ───────────────────────────────────────
        -- payment_value = actual money received
        -- This is the Monetary dimension in RFM
        -- COALESCE handles the small % of orders with no payment record
        COALESCE(p.payment_value, 0.0)     AS payment_value,
        p.payment_type,
        p.payment_installments,

        -- ── SATISFACTION SIGNAL ───────────────────────────
        -- review_score will be null where no review exists
        -- We keep nulls intentionally — dropping them would bias
        -- the avg_review_score calculation in Gold
        r.review_score,

        -- ── ORDER ITEM METRICS ────────────────────────────
        -- item_count = how many products in this order
        -- total_item_price = sum of product prices
        -- total_freight = total shipping cost
        COALESCE(i.item_count,        1)   AS item_count,
        COALESCE(i.total_item_price,  0.0) AS total_item_price,
        COALESCE(i.total_freight,     0.0) AS total_freight,

        -- ── LINEAGE ───────────────────────────────────────
        -- Records when this row was created in Silver
        -- Useful for debugging: tells you which pipeline run produced it
        CURRENT_TIMESTAMP()                AS _silver_processed_at

    FROM brazilian.bronze.olist_orders o

    -- JOIN 1: customers
    -- INNER JOIN because every order must have a customer record
    -- This join gives us customer_unique_id — the critical deduplication
    INNER JOIN brazilian.bronze.olist_customers c
        ON o.customer_id = c.customer_id

    -- JOIN 2: payments
    -- Aggregated BEFORE joining because one order can have multiple
    -- payment rows (installment payments, split payment methods)
    -- SUM gives total amount paid, MAX gives dominant payment type
    LEFT JOIN (
        SELECT
            order_id,
            SUM(payment_value)        AS payment_value,
            MAX(payment_type)         AS payment_type,
            MAX(payment_installments) AS payment_installments
        FROM brazilian.bronze.olist_payments
        GROUP BY order_id
    ) p ON o.order_id = p.order_id

    -- JOIN 3: reviews
    -- Aggregated BEFORE joining because rare cases have multiple reviews
    -- AVG handles multiple review scores for one order
    LEFT JOIN (
        SELECT
            order_id,
            ROUND(AVG(review_score), 2) AS review_score
        FROM brazilian.bronze.olist_reviews
        GROUP BY order_id
    ) r ON o.order_id = r.order_id

    -- JOIN 4: order items
    -- Aggregated BEFORE joining because one order has one row per product
    -- COUNT gives items in order, SUM gives total price and freight
    LEFT JOIN (
        SELECT
            order_id,
            COUNT(*)           AS item_count,
            SUM(price)         AS total_item_price,
            SUM(freight_value) AS total_freight
        FROM brazilian.bronze.olist_order_items
        GROUP BY order_id
    ) i ON o.order_id = i.order_id

    -- BUSINESS FILTER: delivered orders only
    -- Reason: cancelled and processing orders do not represent
    -- a completed customer purchase relationship
    -- Including them would make churned customers appear active
    -- and inflate frequency counts for customers who never completed a purchase
    WHERE o.order_status = 'delivered'
""")

row_count = customer_orders_silver.count()
print(f"Silver table built successfully")
print(f"Rows    : {row_count:,} delivered orders")
print(f"Columns : {len(customer_orders_silver.columns)}")
print()
customer_orders_silver.printSchema()

In [0]:
# ── WRITE TO SILVER ───────────────────────────────────────────
#
# WHY saveAsTable instead of just writing files:
# saveAsTable registers the table in Unity Catalog
# Notebook 03 can then reference it as brazilian.silver.customer_orders
# without knowing where the files physically live

(customer_orders_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE))

written_count = spark.table(TARGET_TABLE).count()
print(f"✅ {TARGET_TABLE}")
print(f"   Rows written : {written_count:,}")